# Roq — robust (uncertainty-aware) cost baseline (evaluation)

**Roq** ("Robust Query Optimization", Kamali et al. 2024,
[arXiv:2401.15210](https://arxiv.org/abs/2401.15210),
[github.com/M2oDA-Lab/Roq](https://github.com/M2oDA-Lab/Roq)) is a risk-aware
cost model built around a calibrated predictive distribution, so it populates all
four metric sheets (like Reqo).

Two ingredients define it and are reproduced in `roq/model.py`:

1. **Dual encoder** — a plan-tree **TCNN** branch (tree convolution + max/dynamic
   pooling) concatenated with a query-graph **GNN** branch (message passing +
   mean/max pooling).
2. **Uncertainty decomposition** — a heteroscedastic head trained with Roq's
   Gaussian NLL (`aleatoric_loss`) gives the **aleatoric** variance; **MC-dropout**
   (`mc_samples=10`) gives the **epistemic** variance. They combine by the law of
   total variance: `sigma_log = sqrt(Ud + Um)`. This split is what separates Roq
   from Reqo (aleatoric-only).

It consumes the **same** `build_graph_dataset` tensors as QPPNet / Zero-Shot /
Reqo. Lakehouse caveats (catalog-statistics gap on the query-graph branch, no
BatchNorm, linear mean head over log-runtime) are documented in the module.


In [ ]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from loader.load import load_aligned_plans_and_runs

from uncertainty_prediction.src import *
from uncertainty_prediction.config import *

from uncertainty_prediction.baselines.predictive.common import (
    build_graph_dataset,
    evaluate_gaussian_predictions,
    print_metric_headers_for_excel,
    print_metrics_for_excel,
)
from uncertainty_prediction.baselines.predictive.roq import RoqBaseline

import torch

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
queries_dir = "/mnt/lakehouse-raw-results/tpcds/lakehouse-a/20260222-191819Z/queries"

plans_by_query, runs_by_query, common = load_aligned_plans_and_runs(
    queries_dir=queries_dir,
    run_ids=RUN_IDS,
    collection=COLLECTION_NAME,
    schema=SCHEMA_NAME,
    instance=LAKEHOUSE_INSTANCE_NAME,
    metric=METRIC,
    xcol=XCOL,
    ycol=YCOL,
    parsed_results_root=PARSED_RESULTS_ROOT,
    canon_fn=canon_qid,
    min_runs=1,
    min_points_per_run=2,
    require_cols=(XCOL, YCOL),
)

train_qids, test_qids = split_query_ids(common, seed=SEED, test_frac=TEST_FRAC)
print("n_train:", len(train_qids), "n_test:", len(test_qids))

In [ ]:
# structured plan graphs — identical tensors QPPNet / Zero-Shot / Reqo consume
gdata = build_graph_dataset(
    plans_by_query=plans_by_query, runs_by_query=runs_by_query,
    train_qids=train_qids, test_qids=test_qids, xcol=XCOL, runtime_mode="mean",
)
print("num_ops:", gdata["num_ops"], "| cont_dim:", gdata["cont_dim"])

y_test_log = gdata["y_test_log"]

## Train + evaluate Roq

In [ ]:
roq = RoqBaseline(num_ops=gdata["num_ops"], cont_dim=gdata["cont_dim"],
                  mc_samples=10, device=device, seed=42)
roq.fit(gdata["train_graphs"], gdata["y_train_log"], num_epochs=100, lr=1e-3, verbose=True)

pred = roq.predict_gaussian(gdata["test_graphs"])
roq_metrics = evaluate_gaussian_predictions(pred["mu_log"], pred["sigma_log"], y_test_log)
print_metric_headers_for_excel(roq_metrics)
print_metrics_for_excel(roq_metrics)

## Uncertainty decomposition (Roq-specific)

Roq's headline is the aleatoric / epistemic split. `predict_gaussian` returns both
components alongside the combined `sigma_log`; the combined value is what the
four sheets score, but the decomposition is worth reporting on its own.


In [ ]:
decomp = pd.DataFrame({
    "aleatoric_var": pred["aleatoric_var"],
    "epistemic_var": pred["epistemic_var"],
    "total_sigma_log": pred["sigma_log"],
})
print("mean aleatoric var:", decomp["aleatoric_var"].mean().round(4),
      "| mean epistemic var:", decomp["epistemic_var"].mean().round(4))
decomp.describe().round(4)

## Full metric summary

Roq is Tier-4 (uncertainty-aware): all four sheets are populated. Paste the
tab-separated rows above into the workbook next to Reqo.


In [ ]:
summary = pd.DataFrame({"Roq": roq_metrics}).T
cols = ["mae", "rmse", "median_q_error", "crps", "cov@50", "cov@90", "cov@99",
        "mpiw", "unc_spearman", "unc_pearson"]
summary.reindex(columns=[c for c in cols if c in roq_metrics])